In [1]:
pip install -U transformers accelerate datasets evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 105.3 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.3/512.3 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.4.1
    Uninstalling datasets-4.4.1:
      Successfully uninstalled datasets-4.4.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
Note: you may need to restart the kernel to use updated packages.


## Local Inference on GPU 
Model page: https://huggingface.co/aaditya/Llama3-OpenBioLLM-8B

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/aaditya/Llama3-OpenBioLLM-8B)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [3]:
model_id = "aaditya/Llama3-OpenBioLLM-8B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Handle missing PAD token (common for LLaMA models)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/449 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
2025-12-19 15:51:46.050952: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766159506.455189      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766159506.572262      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766159507.632090      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766159507.632135      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766159507.632138      55

...

pytorch_model-00004-of-00004.bin:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

pytorch_model-00001-of-00004.bin:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
  

In [14]:
def build_baseline_prompt(symptoms_text):
    return f"""
You are a clinical decision support AI.

Task:
Based on the symptoms below, list the top 3 most likely medical diagnoses and give a brief medical reason for each.

Symptoms:
{symptoms_text}

Answer now.
""".strip()


In [52]:
def generate_single_json(case_text):
    prompt = build_single_json_prompt(case_text)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            min_new_tokens=40,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # Return only the generated part
    return decoded[len(prompt):].strip()


In [6]:
# Absolute minimal generation test
inputs = tokenizer("Hello doctor,", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Hello doctor, I have been experiencing frequent episodes of joint pain and stiffness which last for a couple of hours. I


In [9]:
inputs = tokenizer(
    "Explain what arthritis is in simple terms.",
    return_tensors="pt"
).to(model.device)

out = model.generate(
    **inputs,
    max_new_tokens=128,
    min_new_tokens=20,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

print(tokenizer.decode(out[0], skip_special_tokens=True))



Explain what arthritis is in simple terms. Arthritis is a condition where there is inflammation of the joints, causing pain, swelling, and stiffness. It can be caused by various factors such as infection, autoimmune diseases, or wear and tear over time.


In [16]:
single_test_case = {
    "case_id": "TEST_001",
    "symptoms": [
        "fever",
        "productive cough",
        "shortness of breath",
        "chest pain"
    ],
    "ground_truth": "pneumonia",
    "clinician_reasoning": [
        "fever → infection",
        "productive cough → lung involvement",
        "shortness of breath → impaired gas exchange"
    ]
}


In [21]:
symptoms_text = ", ".join(single_test_case["symptoms"])

output = generate_diagnosis(symptoms_text)

print("Symptoms:", symptoms_text)
print("\nModel Output:\n")
print(output)


Symptoms: fever, productive cough, shortness of breath, chest pain

Model Output:

{
  "diagnoses": [
    {
      "name": "Pneumonia",
      "reasoning": "Fever, productive cough, and shortness of breath are common symptoms of pneumonia."
    },
    {
      "name": "Pneumothorax",
      "reasoning": "Chest pain and shortness of breath can be indicative of a pneumothorax."
    },
    {
      "name": "Bronchitis",
      "reasoning": "Fever, productive cough, and shortness of breath are also common symptoms of bronchitis."
    }
  ]
}


In [98]:
def build_json_prompt(case_text):
    return f"""
You are a clinical decision support AI.

Task:
Given the clinical case below, output the top 3 most likely medical diagnoses.

Rules:
- Output JSON only
- No markdown
- No extra text
- Use real medical diagnoses only
- List EXACTLY 3 diagnoses

For each diagnosis:
- Provide a brief medical reasoning and think about why it is correct
- Reasoning should be concise (about 6–7 sentences)
- Avoid unnecessary detail

JSON format (follow exactly):
{{
  "diagnoses": [
    {{
      "name": "",
      "reasoning": ""
    }},
    {{
      "name": "",
      "reasoning": ""
    }},
    {{
      "name": "",
      "reasoning": ""
    }}
  ]
}}

Clinical case:
{case_text}

JSON:
""".strip()


In [22]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("zou-lab/MedCaseReasoning")

README.md:   0%|          | 0.00/897 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/99.3M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/7.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/13092 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/897 [00:00<?, ? examples/s]

In [24]:
print(ds["train"][0])


{'Unnamed: 0': 0, 'pmcid': 'PMC4863315', 'title': 'A diffuse traumatic neuroma in the palate: a case report', 'journal': 'Journal of Medical Case Reports', 'article_link': 'https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4863315/', 'publication_date': '2016-05-11', 'text': 'Background A traumatic neuroma is a hyperplastic lesion caused by trauma or surgery that involves the peripheral nerves and is not considered to be a true neoplasm . It may occur in any part of the body, including the head, neck, gallbladder and thigh . The clinical features of a traumatic neuroma include the formation of a solitary nodule less than 2 cm in diameter, neuralgic pain, tenderness, paresthesias and increased pain on palpation over the lesion [ 2 , 3 ]. The recommended treatment of a traumatic neuroma is simple excision rather than nerve resection or alcohol blocks . In the oral region, a traumatic neuroma is a rare disorder that occurs most commonly at the mental foramen, lower lip, tongue and intra-osseou

In [25]:
def normalize_case(example, idx):
    return {
        "case_id": f"MC_{idx}",
        "symptoms_text": example["case_prompt"],
        "ground_truth": example["final_diagnosis"]
    }

normalized = ds["train"].map(
    normalize_case,
    with_indices=True,
    remove_columns=ds["train"].column_names
)


Map:   0%|          | 0/13092 [00:00<?, ? examples/s]

In [27]:
print(normalized[0])


{'case_id': 'MC_0', 'symptoms_text': 'A 30-year-old man presented with several months of pain, tenderness, and swelling on the left side of his palate. On examination, the left palatine mucosa was diffusely swollen—most pronounced in the molar region—with poorly defined borders and increased pain on palpation. T2-weighted MRI demonstrated a high-signal-intensity lesion in that area. Panoramic radiography and CT scans of the maxilla and sinuses were unremarkable, and all left maxillary teeth appeared healthy. His medical and family histories were unremarkable, and he denied any history of trauma or inflammation in the head or neck. Empiric treatment with oral antibiotics and nonsteroidal anti-inflammatory drugs for 7 days produced no improvement. Based on the clinical and radiographic findings, a soft-tissue tumor was suspected, and an incisional biopsy was performed.', 'ground_truth': 'Traumatic neuroma'}


In [28]:
split = normalized.train_test_split(test_size=100, seed=42)

test_cases = split["test"]


In [77]:
len(test_cases)


10

In [93]:
baseline_results = []

for i, case in enumerate(test_cases):
    print(f"Running case {i+1}/100")

    raw_output = generate_single_json(case["symptoms_text"])

    diagnoses = extract_diagnoses(raw_output)

    baseline_results.append({
        "case_id": case["case_id"],
        "ground_truth": case["ground_truth"].lower(),
        "raw_output": raw_output,
        "predicted_diagnoses": diagnoses,
        "valid_json": len(diagnoses) == 3
    })


Running case 1/10
Running case 2/10
Running case 3/10
Running case 4/10
Running case 5/10
Running case 6/10
Running case 7/10
Running case 8/10
Running case 9/10
Running case 10/10


In [94]:
valid_count = sum(r["valid_json"] for r in baseline_results)
print(f"Valid diagnosis JSON outputs: {valid_count}/100")


Valid diagnosis JSON outputs: 10/10


In [95]:
def top3_accuracy(results):
    correct = 0

    for r in results:
        gt_norm = normalize_disease(r["ground_truth"])
        preds_norm = [normalize_disease(p) for p in r["predicted_diagnoses"]]

        if gt_norm in preds_norm:
            correct += 1

    return correct / len(results)


In [73]:
import re

def normalize_disease(name):
    """
    Normalize disease names for robust matching.
    """
    if not name:
        return ""

    name = name.lower()
    name = name.replace("’", "'")          # normalize apostrophes
    name = re.sub(r"[^a-z0-9]", "", name)  # remove spaces, hyphens, punctuation
    return name



In [83]:
import re
import json

def extract_diagnoses(raw_text, max_items=3):
    """
    Extract diagnosis names from the 'diagnoses' field of a possibly truncated JSON.
    Returns a list of lowercase diagnosis names.
    """
    # Find the diagnoses array
    match = re.search(
        r'"diagnoses"\s*:\s*\[(.*?)\]',
        raw_text,
        re.DOTALL
    )

    if not match:
        return []

    try:
        # Rebuild minimal valid JSON
        json_str = '{ "diagnoses": [' + match.group(1) + '] }'
        data = json.loads(json_str)

        diagnoses = [
            d.strip().lower()
            for d in data.get("diagnoses", [])
            if isinstance(d, str)
        ]

        return diagnoses[:max_items]

    except json.JSONDecodeError:
        return []


In [97]:
import random
sample = random.choice(baseline_results)
print(sample)


{'case_id': 'MC_5115', 'ground_truth': 'osteochondrolipoma', 'raw_output': '{\n  "diagnoses": ["Non-Hodgkin lymphoma", "Metastatic adenocarcinoma", "Neurofibroma"],\n  "reasoning": {\n    "Non-Hodgkin lymphoma": "The patient\'s presentation of a long-standing foot mass with recent progression, pain, and systemic symptoms such as night sweats and unintentional weight loss, along with imaging findings of a nonhomogeneous mass with calcification, are highly suggestive of Non-Hodgkin lymphoma. The absence of bone involvement on radiographs and the presence of mixed fibrous and bony structures on MRI further support this diagnosis.",\n    "Metastatic adenocarcinoma": "Given the patient\'s age, history of smoking, and the presence of a painful mass with calcification, metastatic adenocarcinoma should be considered. This diagnosis is supported by the mass\'s location and characteristics on imaging, although further investigations would be needed to confirm the primary site of origin.",\n    "

In [39]:
def top3_accuracy(results):
    correct, total = 0, 0

    for r in results:
        if not r["valid_json"]:
            continue

        preds = [d["name"].lower() for d in r["model_output"]["diagnoses"]]
        if r["ground_truth"].lower() in preds:
            correct += 1
        total += 1

    return correct / total

acc = top3_accuracy(baseline_results)
print("Top-3 Accuracy:", acc)


Top-3 Accuracy: 0.061224489795918366


In [40]:
case = next(c for c in test_cases if c["case_id"] == "MC_5609")
raw = generate_diagnosis(case["symptoms_text"])
print(raw)


{
  "diagnoses": [
    {
      "name": "Ewing's sarcoma",
      "reasoning": "The patient's symptoms, including progressive shortness of breath, chest pain, and the presence of a palpable lump in the right postero-inferior chest wall, along with the imaging findings of a mass in the right lower posterior lung field with pleural effusion, suggest a malignant tumor. The working imaging differential of a soft-tissue sarcoma versus a malignant bone tumor is consistent with Ewing's sarcoma, which is a type of bone cancer that typically affects children and young adults and can present with symptoms such as pain, swelling, and limited mobility."
    },
    {
      "name": "Osteosarcoma",
      "reasoning": "Osteosarcoma is another possibility given the patient's age and the presence of a mass that has eroded and destroyed ribs. Osteosarcoma is a type of bone cancer that is known for its aggressive nature and can cause pain, swelling, and limited mobility."
    },
    {
      "name": "Rhabdom

In [100]:
def generate_diagnosis(case_text):
    prompt = build_json_prompt_old_style(case_text)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,     
            min_new_tokens=60,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded[len(prompt):].strip()


In [103]:
import re
import json

def extract_diagnoses(raw_text, max_items=3):
    """
    Extract diagnosis names from old-style JSON:
    {
      "diagnoses": [
        {"name": "...", "reasoning": "..."},
        ...
      ]
    }
    Works even if reasoning is truncated.
    """
    # First try full JSON parse
    try:
        data = json.loads(raw_text)
        names = [
            d["name"].strip().lower()
            for d in data.get("diagnoses", [])
            if "name" in d
        ]
        return names[:max_items]
    except Exception:
        pass

    # Fallback: regex extraction
    matches = re.findall(r'"name"\s*:\s*"([^"]+)"', raw_text)
    return [m.strip().lower() for m in matches[:max_items]]


In [106]:
baseline_results = []

for i, case in enumerate(test_cases):
    print(f"Running case {i+1}/10")

    raw_output = generate_diagnosis(case["symptoms_text"])

    diagnoses = extract_diagnoses(raw_output)

    baseline_results.append({
        "case_id": case["case_id"],
        "ground_truth": case["ground_truth"].lower(),
        "raw_output": raw_output,
        "predicted_diagnoses": diagnoses,
        "valid_json": len(diagnoses) == 3
    })


Running case 1/10
Running case 2/10
Running case 3/10
Running case 4/10
Running case 5/10
Running case 6/10
Running case 7/10
Running case 8/10
Running case 9/10
Running case 10/10


In [107]:
valid_count = sum(r["valid_json"] for r in baseline_results)
print(f"Valid diagnosis JSON outputs: {valid_count}/10")


Valid diagnosis JSON outputs: 10/10


In [108]:
import re

def normalize_disease(name):
    if not name:
        return ""
    name = name.lower()
    name = name.replace("’", "'")
    name = re.sub(r"[^a-z0-9]", "", name)
    return name


In [109]:
def top3_accuracy(results):
    correct = 0

    for r in results:
        gt = normalize_disease(r["ground_truth"])
        preds = [normalize_disease(p) for p in r["predicted_diagnoses"]]

        if gt in preds:
            correct += 1

    return correct / len(results)


In [110]:
acc = top3_accuracy(baseline_results)
print("Top-3 Accuracy:", acc)


Top-3 Accuracy: 0.0


In [113]:
import random
sample = random.choice(baseline_results)

print("GT:", sample["ground_truth"])
print("Preds:", sample["predicted_diagnoses"])
print("Valid JSON:", sample["valid_json"])
print("Raw output:\n", sample["raw_output"][:])


GT: acuteidiopathicmaculopathy
Preds: ['multiple evanescent white dot syndrome', 'acute multifocal placoid pigment epitheliopathy', 'toxoplasmosis']
Valid JSON: True
Raw output:
 {
  "diagnoses": [
    {
      "name": "Multiple evanescent white dot syndrome",
      "reasoning": "The patient's presentation of acute, severe vision loss following an upper respiratory tract infection, along with the fundus findings of cystoid macular edema, serous detachment at the macula, flame-shaped retinal hemorrhages, and juxtafoveal yellowish lesions, are highly suggestive of multiple evanescent white dot syndrome (MEWDS). The absence of systemic symptoms and normal laboratory studies further support this diagnosis. MEWDS is a self-limiting condition that typically resolves within weeks to months without treatment."
    },
    {
      "name": "Acute multifocal placoid pigment epitheliopathy",
      "reasoning": "Acute multifocal placoid pigment epitheliopathy (AMPP) is another differential diagnosis 